# PDF RAG Chatbot using Gemini + LangChain + FAISS

## Features
- Upload multiple PDFs
- Create embeddings using HuggingFace
- Store vectors in FAISS
- Retrieve relevant chunks
- Answer questions using Gemini 2.5 Flash

# **Install Required Libraries**


In [25]:
!pip -q install \
langchain \
langchain-community \
langchain-google-genai \
faiss-cpu \
pypdf

# Enter Your Gemini API Key

## Setup

1. Get a free Gemini API key from Google AI Studio.
2. Run the next cell and paste your API key when prompted.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")

# Load the PDFs

In [27]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

pdf_files = [
    "rag_building1.pdf",
    "rag_building2.pdf"
]

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print(f"Total Pages Loaded: {len(documents)}")

Total Pages Loaded: 34


# Split Documents into Chunks

In [28]:
!pip -q install -U langchain langchain-community langchain-text-splitters

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 144


# Create Embeddings and FAISS Vector Database

In [30]:
!pip -q install sentence-transformers

In [31]:
!pip -q install langchain-huggingface

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print(" Vector database created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Vector database created!


# Load Gemini

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini Loaded!")

Gemini Loaded!


# Create a Retriever

In [34]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":10}
)

print("Retriever Ready!")

Retriever Ready!


# Define the RAG Function

In [35]:
def ask_rag(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(doc.page_content for doc in docs)

    # Prompt for Gemini
    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the information in the provided context.

If the answer is not available in the context, reply:

"I couldn't find that information in the uploaded PDFs."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate response
    response = llm.invoke(prompt)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.content)

    print("\nSOURCES:")
    for doc in docs:
        print(
            f" {doc.metadata['source']} | Page {doc.metadata['page'] + 1}"
        )

    print("=" * 80)

# Start Chatting

In [37]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    ask_rag(question)


Ask a question (type 'exit' to stop): Explain the architecture proposed in the paper.
QUESTION:
Explain the architecture proposed in the paper.

ANSWER:
The Transformer architecture uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.

The encoder is composed of a stack of N = 6 identical layers. Each encoder layer has two sub-layers:
1.  A multi-head self-attention mechanism.
2.  A simple, position-wise fully connected feed-forward network.
Residual connections are employed around each sub-layer, followed by layer normalization. The output of each sub-layer is LayerNorm(x + Sublayer(x)). All sub-layers and embedding layers produce outputs of dimension dmodel = 512. The encoder maps an input sequence of symbol representations to a sequence of continuous representations.

The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers found in each encoder layer, the decoder inserts a third sub-layer:
